In [0]:
# ===========================================
# INSPEKTOR BUDŻET — Notebook 02: Silver
# Cel: Oczyszczenie i przygotowanie danych
# ===========================================

dbutils.widgets.text("projekt", "inspektor_budzet")
dbutils.widgets.text("dostawca", "rowkop")

projekt = dbutils.widgets.get("projekt")
dostawca = dbutils.widgets.get("dostawca")

catalog = projekt
schema = dostawca
sciezka_dostawcy = f"/Volumes/{catalog}/{schema}"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")

print(f"Notebook 02 — Silver 🥈")
print(f"Dostawca: {dostawca} ✅")

In [0]:
# Wczytujemy surowe dane z tabeli bronze
df_bronze = spark.table(f"{catalog}.{schema}.bronze_usage_internal")

# Filtrujemy tylko wiersze naszego dostawcy (po NIP - bardziej wiarygodne niż nazwa)
df_rowkop = df_bronze.filter(df_bronze.NIP_DOSTAWCY == 5512233444)

print(f"Wszystkich wierszy w bronze: {df_bronze.count()}")
print(f"Wierszy dla dostawcy RowKop: {df_rowkop.count()}")
display(df_rowkop)

In [0]:
from pyspark.sql import functions as F

# Sumujemy metry wykopu dla RowKop
df_wykop = (df_rowkop
    .filter(F.col("TYP_ROBOTY") == "WYKOP_ROWU")
    .agg(F.sum("ILOSC").alias("erp_ilosc"))
    .withColumn("position_id", F.lit("WYKOP_ROWU"))
    .withColumn("jednostka", F.lit("mb"))
)

display(df_wykop)

In [0]:
df_operator = (df_rowkop
    .filter(F.col("TYP_ROBOTY") == "PRACA_OPERATORA")
    .agg(F.sum("ILOSC").alias("erp_ilosc"))
    .withColumn("position_id", F.lit("PRACA_OPERATORA"))
    .withColumn("jednostka", F.lit("godz."))
)

display(df_operator)

In [0]:
df_trudny_grunt = (df_rowkop
    .filter(F.col("TYP_ROBOTY") == "WYKOP_ROWU")
    .filter(
        F.col("KATEGORIA_GRUNTU").contains("II") | 
        F.col("KATEGORIA_GRUNTU").contains("III")
    )
    .select("DATA", "LOKALIZACJA", "KATEGORIA_GRUNTU", "PROTOKOL_NR")
)

print(f"Dni z potwierdzonym trudnym gruntem: {df_trudny_grunt.count()}")
display(df_trudny_grunt)

In [0]:
# Zliczamy potwierdzone dni trudnego gruntu
ilosc_trudny_grunt = df_trudny_grunt.count()

# Tworzymy wiersze dla ryczałtu i dodatku
df_ryczalt = spark.createDataFrame([
    (1, "RYCZALT", "m-c")
], ["erp_ilosc", "position_id", "jednostka"])

df_dodatek = spark.createDataFrame([
    (ilosc_trudny_grunt, "DODATEK_TRUDNY_GRUNT", "dzień")
], ["erp_ilosc", "position_id", "jednostka"])

# Łączymy wszystkie 4 pozycje w jedną tabelę
df_silver_summary = (df_wykop
    .union(df_operator)
    .union(df_ryczalt)
    .union(df_dodatek)
)

display(df_silver_summary)

In [0]:
# Zapisujemy tabelę główną
df_silver_summary.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_summary")
print(f"Zapisano: {catalog}.{schema}.silver_summary ✅")

# Zapisujemy szczegółowe dni trudnego gruntu
df_trudny_grunt.write.mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_trudny_grunt_dni")
print(f"Zapisano: {catalog}.{schema}.silver_trudny_grunt_dni ✅")

In [0]:
# Sprawdzam czy tabela silver_summary istnieje i wyświetlam zawartość
df_check = spark.table(f"{catalog}.{schema}.silver_summary")
print(f"Tabela {catalog}.{schema}.silver_summary istnieje ✅")
print(f"Liczba wierszy: {df_check.count()}")
print("\nZawartość tabeli:")
display(df_check)